In [ ]:


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------

import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent

if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))

from utils.export_utils import exportar_csv

PROJECT_ROOT = Path(__file__).resolve().parents[3]
OUTPUT_DIR = PROJECT_ROOT / "scripts" / "Analytics" / "outputs" / "gold_07"


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 07 - Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e Inteligência Artificial?
# ---------------------------------------------------------------------

caminho_gold_07 = (
    PROJECT_ROOT
    / "Gold"
    / "perguntas_negocio"
    / "gold_07_oportunidades_desafios"
)

arquivos_gold_07 = [
    str(arquivo) for arquivo in caminho_gold_07.glob("part-*.csv")
]

print("\nARQUIVOS ENCONTRADOS:")
print(arquivos_gold_07)

if not arquivos_gold_07:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_07}"
    )

df_gold_07 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_07)
)


# ---------------------------------------------------------------------
# CENÁRIO ATUAL - 2025-2026
"""
O cenário atual é isolado antes das comparações históricas para mostrar quais fatores concentram maior percentual de adoção em cada categoria na edição mais recente. O ranking é calculado separadamente por categoria, evitando comparar diretamente indicadores com públicos elegíveis diferentes.
"""
# ---------------------------------------------------------------------

edicao_atual = "2025-2026"

janela_ranking = (
    Window
    .partitionBy("categoria")
    .orderBy(F.desc("pct_adocao"))
)

cenario_atual = (
    df_gold_07
    .filter(F.col("edicao") == edicao_atual)
    .withColumn(
        "ranking",
        F.row_number().over(janela_ranking)
    )
    .select(
        "categoria",
        "ranking",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy(
        "categoria",
        "ranking"
    )
)

print("\n" + "=" * 100)
print("1. CENÁRIO ATUAL - 2025-2026")
print("=" * 100)

cenario_atual.show(
    100,
    truncate=False
)
"""
Os resultados de 2025-2026 mostram quatro frentes distintas: atração de profissionais, insatisfação, desafios de gestão e barreiras para IA. Como cada categoria possui um número próprio de elegíveis, a leitura deve priorizar o percentual de adoção dentro de cada dimensão.
"""


# ---------------------------------------------------------------------
# PRINCIPAIS CRITÉRIOS PARA ESCOLHER UM EMPREGO
"""
O recorte de critérios de escolha de emprego identifica os fatores mais valorizados pelos profissionais no cenário atual e ajuda a traduzir os resultados em oportunidades de atração e proposta de valor ao colaborador.
"""
# ---------------------------------------------------------------------

criterios_emprego_atual = (
    cenario_atual
    .filter(
        F.col("categoria") == "criterios_para_escolher_emprego"
    )
)

print("\n" + "=" * 100)
print("2. CRITÉRIOS PARA ESCOLHER UM EMPREGO - 2025-2026")
print("=" * 100)

criterios_emprego_atual.show(
    20,
    truncate=False
)
"""
Remuneração aparece como o principal critério, com 83,2%, seguida por flexibilidade de trabalho remoto, com 56,6%. Plano de carreira e oportunidades de crescimento ocupa a terceira posição, com 32,3%, indicando que fatores financeiros, flexibilidade e desenvolvimento concentram as maiores taxas de seleção.
"""


# ---------------------------------------------------------------------
# PRINCIPAIS MOTIVOS DE INSATISFAÇÃO PROFISSIONAL
"""
Os motivos de insatisfação são analisados separadamente dos critérios de escolha porque representam fatores de permanência e experiência após a entrada na empresa, e não apenas elementos de atração.
"""
# ---------------------------------------------------------------------

insatisfacao_atual = (
    cenario_atual
    .filter(
        F.col("categoria") == "motivo_insatisfacao_profissional"
    )
)

print("\n" + "=" * 100)
print("3. MOTIVOS DE INSATISFAÇÃO PROFISSIONAL - 2025-2026")
print("=" * 100)

insatisfacao_atual.show(
    20,
    truncate=False
)
"""
Em 2025-2026, remuneração também lidera a insatisfação profissional, com 45,3%, seguida por oportunidades de crescimento, com 34,9%, e oportunidade de aprendizado e trabalho com referências, com 31,5%. O resultado reforça a relevância de carreira, desenvolvimento e remuneração também na retenção.
"""


# ---------------------------------------------------------------------
# PRINCIPAIS DESAFIOS DOS GESTORES
"""
A análise dos gestores busca identificar obstáculos operacionais e organizacionais enfrentados pelas lideranças de Dados, complementando a leitura de atração e retenção com desafios de execução e gestão.
"""
# ---------------------------------------------------------------------

desafios_gestores_atual = (
    cenario_atual
    .filter(
        F.col("categoria") == "desafios_como_gestor"
    )
)

print("\n" + "=" * 100)
print("4. DESAFIOS DOS GESTORES - 2025-2026")
print("=" * 100)

desafios_gestores_atual.show(
    20,
    truncate=False
)
"""
Os principais desafios em 2025-2026 são dividir o tempo entre entregas técnicas e gestão, com 36,3%, gerenciar expectativas das áreas, com 33,3%, e conduzir projetos multidisciplinares, com 29,1%. Os resultados apontam maior concentração em desafios de coordenação e gestão do que em dificuldades estritamente técnicas.
"""


# ---------------------------------------------------------------------
# PRINCIPAIS BARREIRAS PARA USO DE IA
"""
As barreiras para IA são tratadas como uma dimensão própria porque representam condições que podem limitar investimentos e adoção tecnológica, mesmo quando existe interesse organizacional no tema.
"""
# ---------------------------------------------------------------------

barreiras_ia_atual = (
    cenario_atual
    .filter(
        F.col("categoria") == "motivos_para_nao_usar_ia"
    )
)

print("\n" + "=" * 100)
print("5. BARREIRAS PARA USO DE IA - 2025-2026")
print("=" * 100)

barreiras_ia_atual.show(
    20,
    truncate=False
)
"""
Na edição atual, falta de expertise ou recursos lidera com 38,8%, seguida por dados não preparados para IA generativa, com 36,8%, e falta de compreensão dos casos de uso, com 32,0%. ROI não comprovado e segurança e privacidade também permanecem entre as cinco principais barreiras.
"""


# ---------------------------------------------------------------------
# TOP 5 POR DIMENSÃO - CENÁRIO ATUAL
"""
O Top 5 resume cada dimensão para reduzir o volume de indicadores e priorizar os fatores mais relevantes no storytelling executivo, mantendo o ranking calculado dentro de cada categoria.
"""
# ---------------------------------------------------------------------

top_5_atual = (
    cenario_atual
    .filter(F.col("ranking") <= 5)
    .orderBy(
        "categoria",
        "ranking"
    )
)

print("\n" + "=" * 100)
print("6. TOP 5 POR DIMENSÃO - 2025-2026")
print("=" * 100)

top_5_atual.show(
    100,
    truncate=False
)


# ---------------------------------------------------------------------
# EVOLUÇÃO 2024-2025 X 2025-2026
"""
A comparação geral utiliza 2024-2025 e 2025-2026 porque essas duas edições possuem cobertura comum para as quatro categorias analisadas. Somente opções presentes nos dois períodos permanecem no comparativo, evitando calcular variação a partir de valores ausentes.
"""
# ---------------------------------------------------------------------

historico_comparavel = (
    df_gold_07
    .filter(
        F.col("edicao").isin(
            "2024-2025",
            "2025-2026"
        )
    )
)

comparativo_historico = (
    historico_comparavel
    .groupBy(
        "categoria",
        "opcao"
    )
    .pivot(
        "edicao",
        [
            "2024-2025",
            "2025-2026"
        ]
    )
    .agg(F.first("pct_adocao"))
    .withColumnRenamed(
        "2024-2025",
        "pct_2024_2025"
    )
    .withColumnRenamed(
        "2025-2026",
        "pct_2025_2026"
    )
    .filter(
        F.col("pct_2024_2025").isNotNull()
        & F.col("pct_2025_2026").isNotNull()
    )
    .withColumn(
        "variacao_pp",
        F.round(
            F.col("pct_2025_2026") - F.col("pct_2024_2025"),
            1
        )
    )
    .orderBy(
        "categoria",
        F.desc("variacao_pp")
    )
)

print("\n" + "=" * 100)
print("7. EVOLUÇÃO 2024-2025 X 2025-2026")
print("=" * 100)

comparativo_historico.show(
    100,
    truncate=False
)
"""
A variação em pontos percentuais permite identificar mudanças de prioridade entre as duas edições. Entre os maiores avanços estão gestão de projetos multidisciplinares (+8,2 p.p.), dados não preparados para IA (+7,1 p.p.) e dividir o tempo entre entregas técnicas e gestão (+5,8 p.p.).
"""


# ---------------------------------------------------------------------
# MAIORES AUMENTOS ENTRE 2024-2025 E 2025-2026
"""
O recorte de aumentos mantém apenas variações positivas para destacar temas que ganharam relevância entre as duas edições, independentemente da posição absoluta ocupada no ranking atual.
"""
# ---------------------------------------------------------------------

maiores_aumentos = (
    comparativo_historico
    .filter(F.col("variacao_pp") > 0)
    .orderBy(F.desc("variacao_pp"))
)

print("\n" + "=" * 100)
print("8. MAIORES AUMENTOS - 2024-2025 X 2025-2026")
print("=" * 100)

maiores_aumentos.show(
    30,
    truncate=False
)


# ---------------------------------------------------------------------
# MAIORES REDUÇÕES ENTRE 2024-2025 E 2025-2026
"""
As reduções são analisadas separadamente para identificar fatores que perderam peso relativo no período. Entre as maiores quedas aparecem garantir ROI em projetos de dados (-11,6 p.p.), contratar talentos (-8,1 p.p.) e convencer a empresa a aumentar investimentos (-8,1 p.p.).
"""
# ---------------------------------------------------------------------

maiores_reducoes = (
    comparativo_historico
    .filter(F.col("variacao_pp") < 0)
    .orderBy("variacao_pp")
)

print("\n" + "=" * 100)
print("9. MAIORES REDUÇÕES - 2024-2025 X 2025-2026")
print("=" * 100)

maiores_reducoes.show(
    30,
    truncate=False
)


# ---------------------------------------------------------------------
# EVOLUÇÃO HISTÓRICA DAS BARREIRAS PARA IA
"""
As barreiras para IA possuem dados nas três edições e, por isso, recebem uma leitura histórica mais longa. O acompanhamento preserva elegíveis e selecionados de cada período, já que o tamanho da amostra varia entre as pesquisas.
"""
# ---------------------------------------------------------------------

barreiras_ia_historico = (
    df_gold_07
    .filter(
        F.col("categoria") == "motivos_para_nao_usar_ia"
    )
    .select(
        "edicao",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy(
        "opcao",
        "edicao"
    )
)

print("\n" + "=" * 100)
print("10. EVOLUÇÃO HISTÓRICA DAS BARREIRAS PARA IA")
print("=" * 100)

barreiras_ia_historico.show(
    100,
    truncate=False
)


# ---------------------------------------------------------------------
# VARIAÇÃO DAS BARREIRAS PARA IA - PRIMEIRA X ÚLTIMA EDIÇÃO
"""
A comparação entre 2023-2024 e 2025-2026 resume a mudança acumulada do período. O maior aumento ocorre em ROI não comprovado, com +13,1 p.p., seguido por dados não preparados, com +6,2 p.p., e falta de expertise ou recursos, com +4,0 p.p.
"""
# ---------------------------------------------------------------------

comparativo_ia = (
    barreiras_ia_historico
    .groupBy("opcao")
    .pivot(
        "edicao",
        [
            "2023-2024",
            "2025-2026"
        ]
    )
    .agg(F.first("pct_adocao"))
    .withColumnRenamed(
        "2023-2024",
        "pct_2023_2024"
    )
    .withColumnRenamed(
        "2025-2026",
        "pct_2025_2026"
    )
    .withColumn(
        "variacao_pp",
        F.round(
            F.col("pct_2025_2026") - F.col("pct_2023_2024"),
            1
        )
    )
    .orderBy(F.desc("variacao_pp"))
)

print("\n" + "=" * 100)
print("11. BARREIRAS PARA IA - 2023-2024 X 2025-2026")
print("=" * 100)

comparativo_ia.show(
    30,
    truncate=False
)
"""
Ao mesmo tempo, algumas barreiras perdem participação: propriedade intelectual recua 5,5 p.p., falta de compreensão dos casos de uso 4,8 p.p., incerteza regulatória 4,6 p.p. e segurança e privacidade 4,0 p.p. A combinação sugere mudança do foco de parte das preocupações para capacidade de execução, preparação dos dados e comprovação de retorno.
"""


# ---------------------------------------------------------------------
# SÍNTESE EXECUTIVA DOS PRINCIPAIS PONTOS
"""
A síntese executiva reutiliza o Top 5 atual de cada dimensão para consolidar, em uma única saída, os principais sinais de atração, retenção, gestão e adoção de IA sem criar novos indicadores ou pesos.
"""
# ---------------------------------------------------------------------

sintese_executiva = (
    top_5_atual
    .select(
        "categoria",
        "ranking",
        "opcao",
        "pct_adocao"
    )
    .orderBy(
        "categoria",
        "ranking"
    )
)

print("\n" + "=" * 100)
print("12. SÍNTESE EXECUTIVA - TOP 5 DE CADA DIMENSÃO")
print("=" * 100)

sintese_executiva.show(
    100,
    truncate=False
)


# ---------------------------------------------------------------------
# EXPORTAÇÃO DOS RESULTADOS PARA VISUALIZAÇÃO
# ---------------------------------------------------------------------

exportar_csv(
    cenario_atual,
    OUTPUT_DIR,
    "cenario_atual.csv"
)

exportar_csv(
    comparativo_historico,
    OUTPUT_DIR,
    "evolucao_2024_2026.csv"
)

exportar_csv(
    barreiras_ia_historico,
    OUTPUT_DIR,
    "barreiras_ia_historico.csv"
)

exportar_csv(
    comparativo_ia,
    OUTPUT_DIR,
    "barreiras_ia_variacao_historica.csv"
)

exportar_csv(
    sintese_executiva,
    OUTPUT_DIR,
    "sintese_executiva.csv"
)